# **📈YOLOv8**

In [ ]:
!pip install roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="JUEYy9YQeALtsxzh563N")
project = rf.workspace("object-detection-6qhgz").project("urine-sediments-detection-2")
version = project.version(2)

dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 130.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to urine-sediments-detection-2-2 in yolov8:: 100%|██████████| 25346/25346 [00:03<00:00, 6726.00it/s]


In [ ]:
!cat /content/urine-sediments-detection-2-2/data.yaml


names:
- cast
- cryst
- epith
- epithn
- eryth
- leuko
- mycete
nc: 7
roboflow:
  license: CC BY 4.0
  project: urine-sediments-detection-2
  url: https://universe.roboflow.com/object-detection-6qhgz/urine-sediments-detection-2/dataset/2
  version: 2
  workspace: object-detection-6qhgz
test: ../test/images
train: ../train/images
val: ../valid/images


In [ ]:
CRYSTAL_CLASS_IDS = [1]   # add more if needed


In [ ]:
import os
import shutil

BASE_DIR = "/content/urine-sediments-detection-2-2"
OUT_DIR = "/content/crystal_dataset"

splits = ["train", "valid", "test"]

for split in splits:
    os.makedirs(f"{OUT_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{OUT_DIR}/{split}/labels", exist_ok=True)


In [ ]:
def filter_split(split):
    img_dir = f"{BASE_DIR}/{split}/images"
    lbl_dir = f"{BASE_DIR}/{split}/labels"

    out_img_dir = f"{OUT_DIR}/{split}/images"
    out_lbl_dir = f"{OUT_DIR}/{split}/labels"

    for lbl_file in os.listdir(lbl_dir):
        lbl_path = os.path.join(lbl_dir, lbl_file)

        with open(lbl_path, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            cls_id = int(line.split()[0])
            if cls_id in CRYSTAL_CLASS_IDS:
                # remap class id → 0
                parts = line.strip().split()
                parts[0] = "0"
                new_lines.append(" ".join(parts) + "\n")

        if len(new_lines) > 0:
            # save label
            with open(f"{out_lbl_dir}/{lbl_file}", "w") as f:
                f.writelines(new_lines)

            # copy image
            img_file = lbl_file.replace(".txt", ".jpg")
            shutil.copy(
                f"{img_dir}/{img_file}",
                f"{out_img_dir}/{img_file}"
            )

for split in splits:
    filter_split(split)


In [ ]:
data_yaml = f"""
path: {OUT_DIR}
train: train/images
val: valid/images
test: test/images

names:
  0: crystal
"""

with open(f"{OUT_DIR}/data.yaml", "w") as f:
    f.write(data_yaml)


In [ ]:
!ls /content/crystal_dataset/train/images | wc -l
!ls /content/crystal_dataset/train/labels | wc -l


1709
1709


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/crystal_dataset /content/drive/MyDrive/


Mounted at /content/drive


In [ ]:
!pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 67.9 MB/s eta 0:00:00


In [ ]:
!ls /content/drive/MyDrive/crystal_dataset
!cat /content/drive/MyDrive/crystal_dataset/data.yaml


data.yaml  test  train	valid

path: /content/crystal_dataset
train: train/images
val: valid/images
test: test/images

names:
  0: crystal


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import yaml

# Path to the data.yaml file in Google Drive
data_yaml_path = "/content/drive/MyDrive/crystal_dataset/data.yaml"

# Load the existing data.yaml
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update the 'path' in the data_config to point to the correct location in Google Drive
data_config['path'] = "/content/drive/MyDrive/crystal_dataset"

# Save the updated data_config back to the data.yaml file in Google Drive
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_config, f)

results = model.train(
    data=data_yaml_path, # Use the updated data.yaml path
    epochs=80,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.0005,
    weight_decay=0.01,
    cos_lr=True,
    patience=30,
    device=0,
    project="Crystal_Detection_Results",
    name="yolov8m_crystal_v1",
    pretrained=True
)

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/crystal_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8m_crystal_v15, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, per

In [ ]:
model.predict(
    source="/content/drive/MyDrive/crystal_dataset/test/images",
    conf=0.25,
    save=True
)



image 1/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00590_jpg.rf.2de8871f7a0ddf43ef7ebca54adcc3d7.jpg: 640x640 3 crystals, 37.0ms
image 2/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00591_jpg.rf.d2757533f69500bc4325dedd00c7e30f.jpg: 640x640 4 crystals, 37.0ms
image 3/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00592_jpg.rf.3bb08c6953eef14f47160088a661480a.jpg: 640x640 2 crystals, 37.0ms
image 4/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00595_jpg.rf.d6ad95fe0ca9b6f99965d94382dfe880.jpg: 640x640 3 crystals, 37.0ms
image 5/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00610_jpg.rf.5e545e819deada23e4564b52f702c876.jpg: 640x640 4 crystals, 37.0ms
image 6/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00612_jpg.rf.f1db4e2b39dc01195bd5a440425ce982.jpg: 640x640 2 crystals, 37.1ms
image 7/77 /content/drive/MyDrive/crystal_dataset/test/images/nh00614_jpg.rf.761e8444ea77e8f9f336f1543ee60855.jpg: 640x640 1 crystal, 37.0ms
image 

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'crystal'}
 obb: None
 orig_img: array([[[117, 120, 111],
         [117, 120, 111],
         [121, 124, 115],
         ...,
         [103,  94,  91],
         [103,  94,  91],
         [102,  93,  90]],
 
        [[123, 126, 117],
         [123, 126, 117],
         [126, 129, 120],
         ...,
         [106,  97,  94],
         [111, 102,  99],
         [112, 103, 100]],
 
        [[134, 137, 128],
         [134, 137, 128],
         [135, 138, 129],
         ...,
         [107,  98,  95],
         [114, 105, 102],
         [117, 108, 105]],
 
        ...,
 
        [[183, 188, 161],
         [178, 183, 156],
         [188, 192, 167],
         ...,
         [199, 202, 180],
         [202, 205, 183],
         [215, 218, 196]],
 
        [[181, 186, 157],
         [178, 183, 156],
         [189, 193, 168],
         ...,
         [192, 19

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/Crystal_Detection_Results /content/drive/MyDrive/Urine_Crystal_Results/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving c6.jpg to c6.jpg
Saving c5.jpg to c5.jpg
Saving c4.jpg to c4.jpg
Saving c3.jpg to c3.jpg
Saving c2.jpg to c2.jpg


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/Crystal_Detection_Results/yolov8m_crystal_v15/weights/best.pt")


In [ ]:
results = model.predict(
    source="/content/c6.jpg",
    conf=0.25,
    save=True,
    imgsz=640
)


image 1/1 /content/c6.jpg: 640x480 2 crystals, 51.5ms
Speed: 2.5ms preprocess, 51.5ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /content/runs/detect/predict4


# **📈YOLOv11 (Research Comparison)**

In [ ]:
from ultralytics import YOLO

# Load the YOLO11 model
# We use 'yolo11m.pt' to compare with 'yolov8m.pt'
model_11 = YOLO("yolo11m.pt")

# Train the model using the same dataset and hyperparameters
results_11 = model_11.train(
    data="/content/drive/MyDrive/crystal_dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.0005,
    weight_decay=0.01,
    cos_lr=True,
    patience=30,
    device=0,
    project="Crystal_Detection_Results",
    name="yolo11m_crystal_v1",
    pretrained=True
)

Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/crystal_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo11m_crystal_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=30, pers

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

!cp -r /content/Crystal_Detection_Results /content/drive/MyDrive/Urine_Crystal_Results/

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/Crystal_Detection_Results/yolo11m_crystal_v1/weights/best.pt")

In [ ]:
results = model.predict(
    source="/content/uc.jpg",
    conf=0.25,
    save=True,
    imgsz=640
)


image 1/1 /content/uc.jpg: 640x640 4 crystals, 37.7ms
Speed: 2.4ms preprocess, 37.7ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict5
